In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=47fb23105381606070538c52b81a85dcde347b8f79935d01399b26931312111e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
)

In [4]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ml_projects/NLP_Text_Summarization"
)

RAW_DATA_DIR = PROJECT_ROOT / "artifacts/data/raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "artifacts/data/processed"
TOKENIZER_DIR = PROJECT_ROOT / "artifacts/tokenizer"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("TOKENIZER_DIR:", TOKENIZER_DIR)

RAW_DATA_DIR: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw
PROCESSED_DATA_DIR: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/processed
TOKENIZER_DIR: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/tokenizer


In [5]:
train_path = RAW_DATA_DIR / "samsum_train.json"
val_path = RAW_DATA_DIR / "samsum_validation.json"
test_path = RAW_DATA_DIR / "samsum_test.json"

print(train_path)
print(val_path)
print(test_path)

/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_train.json
/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_validation.json
/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/raw/samsum_test.json


In [6]:
data_files = {
    "train": str(train_path),
    "validation": str(val_path),
    "test": str(test_path),
}
dataset = load_dataset(
    "json",
    data_files=data_files,
)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [7]:
for split in dataset:
    print(f"{split} size: {len(dataset[split])}")

train size: 14731
validation size: 818
test size: 819


In [8]:
sample = dataset["train"][0]

print("Dialogue:\n")
print(sample["dialogue"])

print("\nSummary:\n")
print(sample["summary"])

Dialogue:

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:

Amanda baked cookies and will bring Jerry some tomorrow.


In [9]:
MODEL_NAME = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

PegasusTokenizer(name_or_path='google/pegasus-large', vocab_size=96000, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask_2>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<n>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	96000: AddedToken("<mask_2>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [10]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64

In [11]:
def preprocess_function(examples):

    # Tokenize input dialogues
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    # Tokenize target summaries
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    # ========================================================}
    # Replace PAD token IDs with -100
    # so they are ignored during loss computation
    # ========================================================}
    processed_labels = []
    for label_sequence in labels["input_ids"]:
        processed_sequence = [
            token if token != tokenizer.pad_token_id else -100
            for token in label_sequence
        ]
        processed_labels.append(processed_sequence)
    model_inputs["labels"] = processed_labels
    return model_inputs

In [12]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [13]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [14]:
sample_tokenized = tokenized_dataset["train"][0]
print(sample_tokenized.keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [15]:
print("Input IDs Length:", len(sample_tokenized["input_ids"]))
print("Attention Mask Length:", len(sample_tokenized["attention_mask"]))
print("Labels Length:", len(sample_tokenized["labels"]))

Input IDs Length: 512
Attention Mask Length: 512
Labels Length: 64


In [16]:
decoded_dialogue = tokenizer.decode(
    sample_tokenized["input_ids"],
    skip_special_tokens=True,
)
print(decoded_dialogue)

Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)


In [17]:
cleaned_labels = [
    token
    for token in sample_tokenized["labels"]
    if token != -100
]
decoded_summary = tokenizer.decode(
    cleaned_labels,
    skip_special_tokens=True,
)
print(decoded_summary)

Amanda baked cookies and will bring Jerry some tomorrow.


In [18]:
processed_train_path = PROCESSED_DATA_DIR / "tokenized_train"
processed_val_path = PROCESSED_DATA_DIR / "tokenized_validation"
processed_test_path = PROCESSED_DATA_DIR / "tokenized_test"

In [19]:
tokenized_dataset["train"].save_to_disk(processed_train_path)
tokenized_dataset["validation"].save_to_disk(processed_val_path)
tokenized_dataset["test"].save_to_disk(processed_test_path)

print("Processed datasets saved successfully.")

Saving the dataset (0/1 shards):   0%|          | 0/14731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Processed datasets saved successfully.


In [20]:
tokenizer.save_pretrained(TOKENIZER_DIR)
print("Tokenizer saved successfully.")

Tokenizer saved successfully.


In [21]:
print("=" * 60)
print("Notebook 03 Completed Successfully")
print("=" * 60)

print(f"Train Samples      : {len(tokenized_dataset['train'])}")
print(f"Validation Samples : {len(tokenized_dataset['validation'])}")
print(f"Test Samples       : {len(tokenized_dataset['test'])}")

print(f"Max Input Length   : {MAX_INPUT_LENGTH}")
print(f"Max Target Length  : {MAX_TARGET_LENGTH}")

Notebook 03 Completed Successfully
Train Samples      : 14731
Validation Samples : 818
Test Samples       : 819
Max Input Length   : 512
Max Target Length  : 64
